# Projet Éthique de l'IA : Adult Income

**Cas d'usage :** pré-qualification automatisée d'une demande de **crédit à la consommation**. Une banque fictive veut savoir si le revenu annuel d'un demandeur dépasse 50 000 $ pour l'orienter vers une offre de crédit. Il s'agit donc d'une **aide à la décision** (cas « unbiased prediction », slide 28 du cours 4) : on audite **et** on réduit les biais.

**Question :** peut-on prédire le revenu tout en limitant les écarts de traitement entre groupes (sexe, origine déclarée, et leur **croisement**) ?

Ce notebook :

1. audite les biais historiques par sexe, par origine et par **sexe × origine** (analyse intersectionnelle) ;
2. applique une repondération (Kamiran–Calders) sur le groupe intersectionnel ;
3. adapte un modèle de langage avec les **trois méthodes du cours 3** : fine-tuning complet, LoRA, distillation ;
4. mesure performance, équité, temps, nombre de paramètres entraînés et taille sur disque ;
5. **retient** le modèle le plus adapté selon un critère explicite ;
6. l'utilise sur une **démo contrefactuelle** et explique ses décisions avec **LIME** et **SHAP**.

> Usage pédagogique uniquement : ce prototype ne doit pas servir à prendre une décision réelle sur une personne.

## 0. Ethics by design (cours 4)

**Parties prenantes impactées**

| Partie prenante | Impact possible |
|---|---|
| Demandeurs de crédit | Refus injustifié, accès réduit au crédit |
| Groupes historiquement défavorisés (femmes, minorités) | Reproduction automatisée des inégalités passées |
| Conseillers bancaires | Confiance excessive dans un score « objectif » |
| Banque | Risque juridique (AI Act, RGPD) et réputationnel |
| Société | Boucle de rétroaction : refus → revenus plus faibles → nouveaux refus |

**Red teaming éthique :** on teste le modèle en ne changeant **que** le sexe et l'origine d'un profil (section 17). Si la probabilité change, le modèle utilise l'attribut sensible.

**Cadre réglementaire (cours 1)**

- **AI Act** : l'évaluation de la solvabilité des personnes physiques est classée **à haut risque** (annexe III) : documentation, supervision humaine, explicabilité et gestion des biais obligatoires.
- **RGPD** : l'origine est une donnée sensible (art. 9) ; une personne a le droit de ne pas faire l'objet d'une décision **entièrement** automatisée produisant des effets juridiques (art. 22).

**Protections par défaut :** le modèle produit une **recommandation**, jamais une décision finale ; une explication LIME/SHAP accompagne chaque score ; un recours humain est prévu.

## 1. Installation

Dans Colab : **Exécution → Modifier le type d'exécution → GPU T4**, puis **Tout exécuter**. Si Colab demande de redémarrer la session après l'installation, redémarrer puis relancer toutes les cellules.

In [ ]:
!pip -q install "transformers==4.46.3" "datasets==3.1.0" "accelerate==1.1.1" "peft==0.13.2" "shap==0.46.0" "lime==0.2.0.1" "seaborn==0.13.2"

In [ ]:
import os, json, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, set_seed
)
from peft import LoraConfig, TaskType, get_peft_model

warnings.filterwarnings("ignore")
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

BASE_MODEL = "distilbert-base-uncased"
STUDENT_MODEL = "prajjwal1/bert-tiny"
MAX_LENGTH = 64          # les descriptions font ~40 tokens
N_SAMPLES = 3000
EPOCHS = 3               # 1 epoch ne suffit pas : le modèle prédisait presque toujours <=50K
MIN_GROUP_SIZE = 20      # groupes plus petits exclus des écarts d'équité (trop instables)
RESULTS_DIR = "/content/resultats_projet_ethique"
MODELS_DIR = "/content/modeles_ethique"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil :", device)

## 2. Chargement et préparation des données

Adult Income (recensement américain de 1994) contient des informations socio-économiques et une cible indiquant si le revenu dépasse 50 000 $ par an. Les variables sensibles sont conservées pour **auditer** les écarts, pas pour légitimer leur usage décisionnel.

On crée aussi la variable intersectionnelle **`sex_race`** (ex. `Female | Black`) : un biais peut être invisible quand on regarde le sexe et l'origine séparément, et apparaître seulement sur leur croisement.

In [ ]:
adult = fetch_openml(name="adult", version=2, as_frame=True)
df = adult.data.copy()
df["label"] = adult.target.astype(str).str.contains(">50K", regex=False).astype(int).to_numpy()

for col in df.columns:
    if col != "label":
        df[col] = df[col].astype(str).replace({"?": "Unknown", "nan": "Unknown"})

required = ["age", "workclass", "education", "occupation", "hours-per-week", "sex", "race", "label"]
df = df[required].dropna().reset_index(drop=True)
df["sex_race"] = df["sex"] + " | " + df["race"]
df["stratum"] = df["label"].astype(str) + "_" + df["sex"].astype(str)

sample_n = min(N_SAMPLES, len(df))
# échantillon stratifié : mêmes proportions sexe × cible que le dataset complet
df = (df.groupby("stratum", group_keys=False)
        .sample(frac=sample_n / len(df), random_state=SEED)
        .reset_index(drop=True))

print(df.shape)
print("Taux global >50K :", round(df["label"].mean(), 3))
display(df.head())

## 3. Audit des biais historiques (biais de données, cours 2)

On mesure le taux de cible positive par groupe. Une différence observée n'est pas automatiquement une preuve de discrimination causale, mais constitue un signal à examiner. Les effectifs sont affichés : un taux calculé sur quelques personnes n'est pas fiable.

In [ ]:
def positive_rates(frame, sensitive):
    return (
        frame.groupby(sensitive)["label"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "positive_rate"})
        .sort_values("positive_rate", ascending=False)
    )

sex_audit = positive_rates(df, "sex")
race_audit = positive_rates(df, "race")
display(sex_audit)
display(race_audit)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=df, x="sex", y="label", ax=axes[0], errorbar=None)
axes[0].set_title("Taux >50K par sexe")
axes[0].set_ylabel("Taux positif")
sns.barplot(data=df, x="race", y="label", ax=axes[1], errorbar=None)
axes[1].tick_params(axis="x", rotation=35)
axes[1].set_title("Taux >50K par origine déclarée")
axes[1].set_ylabel("Taux positif")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/audit_biais.png", dpi=160)
plt.show()

### 3.1 Audit intersectionnel : sexe × origine

On croise les deux attributs : chaque cellule donne le taux de >50K **et** l'effectif du groupe.

In [ ]:
inter = (
    df.groupby(["race", "sex"])["label"]
    .agg(positive_rate="mean", count="count")
    .reset_index()
)
inter_table = inter.pivot(index="race", columns="sex", values="positive_rate")
count_table = inter.pivot(index="race", columns="sex", values="count").fillna(0).astype(int)
order = df.groupby("race")["label"].mean().sort_values(ascending=False).index
inter_table, count_table = inter_table.loc[order], count_table.loc[order]

display(inter.sort_values("positive_rate", ascending=False).reset_index(drop=True))

annot = inter_table.map(lambda v: "" if pd.isna(v) else f"{v:.0%}") + "\n(n=" + count_table.astype(str) + ")"

fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [1, 1.4]})
sns.heatmap(inter_table, annot=annot, fmt="", cmap="Blues", vmin=0,
            cbar_kws={"label": "Taux >50K"}, ax=axes[0])
axes[0].set_title("Taux >50K par sexe × origine")
axes[0].set_xlabel("Sexe"); axes[0].set_ylabel("Origine déclarée")

sns.barplot(data=df, x="race", y="label", hue="sex", order=order, errorbar=None, ax=axes[1])
axes[1].axhline(df["label"].mean(), ls="--", c="grey", lw=1, label="Moyenne globale")
axes[1].tick_params(axis="x", rotation=35)
axes[1].set_title("Taux >50K par origine, séparé par sexe")
axes[1].set_ylabel("Taux positif"); axes[1].set_xlabel("")
axes[1].legend(title="Sexe")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/audit_intersectionnel.png", dpi=160)
plt.show()

top = inter[inter["count"] >= MIN_GROUP_SIZE].sort_values("positive_rate")
print(f"Écart entre groupes intersectionnels (n >= {MIN_GROUP_SIZE}) : "
      f"{top.iloc[-1]['sex']} | {top.iloc[-1]['race']} = {top.iloc[-1]['positive_rate']:.0%} "
      f"vs {top.iloc[0]['sex']} | {top.iloc[0]['race']} = {top.iloc[0]['positive_rate']:.0%}")

## 4. Transformation des lignes en texte

Le modèle de langage reçoit une description structurée de chaque personne.

On prépare **deux versions** du texte :

- `text` : avec le sexe et l'origine (version de référence) ;
- `text_blind` : **sans** ces deux attributs (« fairness through unawareness »). Cela permet de vérifier si supprimer l'attribut suffit, ou si le biais passe par des variables corrélées (profession, heures travaillées…).

In [ ]:
def row_to_text(row, blind=False):
    text = (
        f"Age: {row['age']}. Workclass: {row['workclass']}. "
        f"Education: {row['education']}. Occupation: {row['occupation']}. "
        f"Hours per week: {row['hours-per-week']}."
    )
    if not blind:
        text += f" Sex: {row['sex']}. Race: {row['race']}."
    return text

df["text"] = df.apply(row_to_text, axis=1)
df["text_blind"] = df.apply(lambda r: row_to_text(r, blind=True), axis=1)
train_df, test_df = train_test_split(
    df, test_size=0.25, random_state=SEED, stratify=df["stratum"]
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print(len(train_df), len(test_df))
print(train_df.loc[0, "text"])
print(train_df.loc[0, "text_blind"])

## 5. Réduction du biais par repondération

La pondération de **Kamiran–Calders** donne à chaque exemple le poids `P(groupe) × P(cible) / P(groupe, cible)`, ce qui rend la cible statistiquement indépendante du groupe dans les données d'entraînement.

On l'applique au groupe **intersectionnel `sex_race`** (et pas seulement au sexe), pour viser aussi les écarts d'origine. Les poids sont **bornés entre 0,5 et 3** pour éviter qu'un très petit groupe ne domine l'entraînement : c'est une adaptation de la méthode originale. Les poids sont calculés **uniquement sur le jeu d'entraînement** (pas de fuite de données).

In [ ]:
def add_reweighing_weights(frame, sensitive="sex_race", clip=(0.5, 3.0)):
    out = frame.copy()
    n = len(out)
    p_s = out[sensitive].value_counts() / n
    p_y = out["label"].value_counts() / n
    p_sy = out.groupby([sensitive, "label"]).size() / n
    def weight(row):
        denom = p_sy.loc[(row[sensitive], row["label"])]
        return (p_s.loc[row[sensitive]] * p_y.loc[row["label"]]) / denom
    out["sample_weight"] = out.apply(weight, axis=1).clip(*clip)
    return out

train_fair_df = add_reweighing_weights(train_df)
display(
    train_fair_df.groupby(["sex", "race", "label"])["sample_weight"]
    .agg(["mean", "count"]).round(3)
)

## 6. Tokenisation et jeux Hugging Face

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def to_hf(frame, weighted=False, text_col="text"):
    payload = {
        "text": frame[text_col].tolist(),
        "labels": frame["label"].astype(int).tolist(),
        # poids 1 par défaut : toutes les variantes ont les mêmes colonnes
        "sample_weight": (frame["sample_weight"].astype(float).tolist()
                          if weighted else [1.0] * len(frame)),
    }
    ds = Dataset.from_dict(payload)
    return ds.map(
        lambda batch: tokenizer(
            batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH
        ),
        batched=True,
        remove_columns=["text"],
    )

train_ds = to_hf(train_df)
train_fair_ds = to_hf(train_fair_df, weighted=True)
test_ds = to_hf(test_df)
train_fair_blind_ds = to_hf(train_fair_df, weighted=True, text_col="text_blind")
test_blind_ds = to_hf(test_df, text_col="text_blind")

## 7. Métriques de performance et d'équité (biais d'évaluation, cours 2)

L'accuracy seule est trompeuse : avec ~24 % de >50K, un modèle qui répond toujours « <=50K » obtient ~76 % d'accuracy **et** des écarts d'équité nuls. On suit donc aussi :

- **F1 macro** et **recall de la classe >50K** ;
- **taux de prédiction positive** (doit être proche du taux réel, ~24 %) ;
- **Demographic parity gap** : écart maximal des taux de prédiction positive entre groupes ;
- **Equal opportunity gap** : écart maximal des vrais taux positifs (parmi les vrais >50K).

Les écarts sont calculés pour `sex`, `race` et `sex_race`, en excluant les groupes de moins de `MIN_GROUP_SIZE` personnes (et, pour l'equal opportunity, de moins de 10 vrais >50K).

In [ ]:
def classification_metrics(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    _, r_pos, _, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(p),
        "recall_macro": float(r),
        "f1_macro": float(f1),
        "recall_pos": float(r_pos),
        "positive_rate_pred": float(np.mean(y_pred)),
    }

def fairness_gaps(y_true, y_pred, groups, min_size=MIN_GROUP_SIZE):
    tmp = pd.DataFrame({"y": y_true, "pred": y_pred, "group": groups})
    sizes = tmp["group"].value_counts()
    tmp = tmp[tmp["group"].isin(sizes[sizes >= min_size].index)]
    dp = tmp.groupby("group")["pred"].mean()
    pos = tmp[tmp["y"] == 1]
    pos_sizes = pos["group"].value_counts()
    # au moins 10 vrais >50K par groupe pour que le vrai taux positif ait un sens
    eo = pos[pos["group"].isin(pos_sizes[pos_sizes >= 10].index)].groupby("group")["pred"].mean()
    return {
        "demographic_parity_gap": float(dp.max() - dp.min()) if len(dp) > 1 else 0.0,
        "equal_opportunity_gap": float(eo.max() - eo.min()) if len(eo) > 1 else 0.0,
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = logits[0] if isinstance(logits, tuple) else logits
    preds = np.argmax(logits, axis=-1)
    return classification_metrics(labels, preds)

def trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

def dir_size_mb(path):
    return sum(
        os.path.getsize(os.path.join(root, f))
        for root, _, files in os.walk(path) for f in files
    ) / 1e6

SENSITIVE = ["sex", "race", "sex_race"]
PREDICTIONS = {}

def evaluate_trainer(name, trainer, elapsed, model, save_dir, eval_ds=None):
    model.save_pretrained(save_dir)
    output = trainer.predict(eval_ds if eval_ds is not None else test_ds)
    logits = output.predictions[0] if isinstance(output.predictions, tuple) else output.predictions
    pred = np.argmax(logits, axis=-1)
    y = test_df["label"].to_numpy()
    metrics = classification_metrics(y, pred)
    for sensitive in SENSITIVE:
        gaps = fairness_gaps(y, pred, test_df[sensitive].to_numpy())
        metrics.update({f"{sensitive}_{k}": v for k, v in gaps.items()})
    trainable, total = trainable_parameters(model)
    metrics.update({
        "model": name,
        "training_seconds": round(elapsed, 1),
        "trainable_parameters": int(trainable),
        "total_parameters": int(total),
        "disk_size_mb": round(dir_size_mb(save_dir), 2),
    })
    PREDICTIONS[name] = pred
    print(f"{name} | F1 macro={metrics['f1_macro']:.3f} | "
          f"taux prédit >50K={metrics['positive_rate_pred']:.1%} (réel {y.mean():.1%})")
    if metrics["positive_rate_pred"] < 0.02:
        print("  ⚠️ Le modèle ne prédit presque jamais >50K : les écarts d'équité ne sont pas interprétables.")
    return metrics, pred

## 8. Entraîneur pondéré

La perte de chaque exemple est multipliée par son poids, puis normalisée par la **somme** des poids (moyenne pondérée), ce qui garde une échelle de perte stable d'un batch à l'autre.

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs = dict(inputs)
        weights = inputs.pop("sample_weight", None)
        labels = inputs.get("labels")
        outputs = model(**inputs)
        if weights is None:
            loss = outputs.loss
        else:
            per_item = F.cross_entropy(outputs.logits, labels, reduction="none")
            w = weights.to(per_item.device, dtype=per_item.dtype)
            loss = (per_item * w).sum() / w.sum()
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = dict(inputs)
        inputs.pop("sample_weight", None)
        return super().prediction_step(model, inputs, prediction_loss_only, ignore_keys)

def args_for(output_dir, epochs=EPOCHS, lr=2e-5):
    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        learning_rate=lr,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
        remove_unused_columns=False,
    )

def train_and_evaluate(name, model, train_dataset, save_dir, lr, eval_ds=None, trainer_cls=WeightedTrainer, **kw):
    trainer = trainer_cls(
        model=model,
        args=args_for(save_dir, lr=lr),
        train_dataset=train_dataset,
        eval_dataset=eval_ds if eval_ds is not None else test_ds,
        compute_metrics=compute_metrics,
        **kw,
    )
    start = time.time()
    trainer.train()
    elapsed = time.time() - start
    return evaluate_trainer(name, trainer, elapsed, model, save_dir, eval_ds)

## 9. Méthode A : fine-tuning complet (avant / après correction)

Tous les paramètres du modèle sont ajustés. On entraîne une version **sans** correction (poids = 1) et une version **avec** repondération, pour isoler l'effet de la correction.

In [ ]:
metrics_full_raw, _ = train_and_evaluate(
    "Full FT (original)",
    AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2),
    train_ds, f"{MODELS_DIR}/full_raw", lr=2e-5,
)
model_full_fair = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
metrics_full_fair, _ = train_and_evaluate(
    "Full FT (fair)", model_full_fair, train_fair_ds, f"{MODELS_DIR}/full_fair", lr=2e-5,
)

## 10. Méthode B : LoRA (avant / après correction)

LoRA **gèle** le modèle de base et n'entraîne que de petites matrices de rang faible ajoutées aux couches d'attention (ici `q_lin` et `v_lin`), plus la tête de classification.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
)

def new_lora():
    return get_peft_model(
        AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2),
        lora_config,
    )

model_lora_raw = new_lora()
metrics_lora_raw, _ = train_and_evaluate(
    "LoRA (original)", model_lora_raw, train_ds, f"{MODELS_DIR}/lora_raw", lr=2e-4,
)
model_lora_fair = new_lora()
metrics_lora_fair, _ = train_and_evaluate(
    "LoRA (fair)", model_lora_fair, train_fair_ds, f"{MODELS_DIR}/lora_fair", lr=2e-4,
)

### 10.1 Variante : LoRA corrigé **sans** sexe ni origine dans le texte

Même configuration que « LoRA (fair) », mais le modèle ne voit plus ces deux attributs. Les écarts sont toujours mesurés avec les vrais groupes des personnes.

In [ ]:
model_lora_blind = new_lora()
metrics_lora_blind, _ = train_and_evaluate(
    "LoRA (fair, sans attributs)", model_lora_blind, train_fair_blind_ds,
    f"{MODELS_DIR}/lora_blind", lr=2e-4, eval_ds=test_blind_ds,
)

## 11. Méthode C : distillation

Le **LoRA corrigé** devient le professeur. Un petit BERT (`bert-tiny`, ~4 M de paramètres) apprend à reproduire ses probabilités (perte KL avec température) tout en gardant une perte supervisée pondérée. Le professeur et l'élève partagent le même vocabulaire WordPiece (BERT uncased) : on peut donc leur donner les mêmes entrées tokenisées (vérifié ci-dessous).

In [ ]:
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
assert student_tokenizer.get_vocab() == tokenizer.get_vocab(), \
    "Vocabulaires différents : il faut re-tokeniser les données pour l'élève."

class DistillationTrainer(WeightedTrainer):
    def __init__(self, teacher_model=None, temperature=2.0, alpha=0.6, **kwargs):
        super().__init__(**kwargs)
        self.teacher_model = teacher_model.to(device).eval()
        self.temperature = temperature
        self.alpha = alpha
        for p in self.teacher_model.parameters():
            p.requires_grad = False

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs = dict(inputs)
        weights = inputs.pop("sample_weight", None)
        labels = inputs["labels"]
        student_outputs = model(**inputs)
        with torch.no_grad():
            teacher_inputs = {k: v for k, v in inputs.items() if k != "labels"}
            teacher_logits = self.teacher_model(**teacher_inputs).logits

        ce_each = F.cross_entropy(student_outputs.logits, labels, reduction="none")
        if weights is not None:
            w = weights.to(ce_each.device, dtype=ce_each.dtype)
            ce_loss = (ce_each * w).sum() / w.sum()
        else:
            ce_loss = ce_each.mean()

        t = self.temperature
        kd_loss = F.kl_div(
            F.log_softmax(student_outputs.logits / t, dim=-1),
            F.softmax(teacher_logits / t, dim=-1),
            reduction="batchmean",
        ) * (t ** 2)
        loss = self.alpha * kd_loss + (1 - self.alpha) * ce_loss
        return (loss, student_outputs) if return_outputs else loss

student_model = AutoModelForSequenceClassification.from_pretrained(STUDENT_MODEL, num_labels=2)
metrics_student, _ = train_and_evaluate(
    "Distillation (fair)", student_model, train_fair_ds, f"{MODELS_DIR}/student",
    lr=5e-4, trainer_cls=DistillationTrainer, teacher_model=model_lora_fair,
)

## 12. Benchmark final

Les trois méthodes sont entraînées avec le **même nombre d'epochs**. Pour LoRA, la taille sur disque correspond aux seuls adaptateurs (le modèle de base est partagé).

In [ ]:
all_metrics = [metrics_full_raw, metrics_full_fair, metrics_lora_raw,
               metrics_lora_fair, metrics_lora_blind, metrics_student]
results = pd.DataFrame(all_metrics).set_index("model")

cols = ["accuracy", "f1_macro", "recall_pos", "positive_rate_pred",
        "sex_demographic_parity_gap", "race_demographic_parity_gap", "sex_race_demographic_parity_gap",
        "sex_race_equal_opportunity_gap",
        "training_seconds", "trainable_parameters", "disk_size_mb"]
display(results[cols].round(4))
results.to_csv(f"{RESULTS_DIR}/benchmark_modeles.csv")
with open(f"{RESULTS_DIR}/benchmark_modeles.json", "w", encoding="utf-8") as f:
    json.dump(results.reset_index().to_dict(orient="records"), f, indent=2, ensure_ascii=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
plots = [
    ("f1_macro", "F1 macro", (0, 1)),
    ("sex_demographic_parity_gap", "Écart de parité : sexe", None),
    ("sex_race_demographic_parity_gap", "Écart de parité : sexe × origine", None),
    ("training_seconds", "Temps d'entraînement (s)", None),
    ("trainable_parameters", "Paramètres entraînés", None),
    ("disk_size_mb", "Taille sur disque (Mo)", None),
]
for ax, (col, title, ylim) in zip(axes.flat, plots):
    results[col].plot(kind="bar", ax=ax, title=title, ylim=ylim, color="#4C72B0")
    ax.tick_params(axis="x", rotation=35)
    ax.set_xlabel("")
axes[1, 1].set_yscale("log")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/benchmark.png", dpi=160)
plt.show()

## 13. Comparaison avant / après correction

In [ ]:
compare_cols = [
    "accuracy", "f1_macro", "positive_rate_pred",
    "sex_demographic_parity_gap", "sex_equal_opportunity_gap",
    "race_demographic_parity_gap", "race_equal_opportunity_gap",
    "sex_race_demographic_parity_gap", "sex_race_equal_opportunity_gap",
]
before_after = results.loc[
    ["Full FT (original)", "Full FT (fair)", "LoRA (original)", "LoRA (fair)", "LoRA (fair, sans attributs)"],
    compare_cols,
]
display(before_after.round(4))
before_after.to_csv(f"{RESULTS_DIR}/comparaison_avant_apres.csv")

### 13.1 Taux de prédiction >50K par groupe sexe × origine, avant / après

Même lecture que l'audit de la section 3.1, mais sur les **prédictions** du modèle (jeu de test). Les groupes de moins de `MIN_GROUP_SIZE` personnes sont grisés.

In [ ]:
def group_rate_table(pred):
    tmp = test_df[["sex", "race"]].copy()
    tmp["pred"] = pred
    g = tmp.groupby(["race", "sex"])["pred"].agg(["mean", "count"]).reset_index()
    rate = g.pivot(index="race", columns="sex", values="mean")
    n = g.pivot(index="race", columns="sex", values="count").fillna(0).astype(int)
    return rate.loc[order], n.loc[order]

panels = [("Réalité (test)", test_df["label"].to_numpy()),
          ("LoRA original", PREDICTIONS["LoRA (original)"]),
          ("LoRA corrigé", PREDICTIONS["LoRA (fair)"]),
          ("LoRA corrigé sans attributs", PREDICTIONS["LoRA (fair, sans attributs)"])]

fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=True)
for ax, (title, pred) in zip(axes, panels):
    rate, n = group_rate_table(pred)
    annot = rate.map(lambda v: "" if pd.isna(v) else f"{v:.0%}") + "\n(n=" + n.astype(str) + ")"
    sns.heatmap(rate, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=0.6,
                mask=(n < MIN_GROUP_SIZE), cbar=False, ax=ax)
    ax.set_facecolor("#e0e0e0")
    ax.set_title(title); ax.set_xlabel("Sexe"); ax.set_ylabel("")
axes[0].set_ylabel("Origine déclarée")
plt.suptitle("Taux >50K par sexe × origine : réalité vs prédictions", y=1.03)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/intersectionnel_avant_apres.png", dpi=160, bbox_inches="tight")
plt.show()

## 14. Matrices de confusion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, ["LoRA (original)", "LoRA (fair)", "LoRA (fair, sans attributs)"]):
    sns.heatmap(
        confusion_matrix(test_df["label"], PREDICTIONS[name]),
        annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
        xticklabels=["<=50K", ">50K"], yticklabels=["<=50K", ">50K"],
    )
    ax.set_title(name)
    ax.set_xlabel("Prédiction")
    ax.set_ylabel("Vérité")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/confusions_avant_apres.png", dpi=160)
plt.show()

## 15. Choix du modèle retenu

Critère (cours 3, slide « Benchmark ») : pour un usage bancaire à haut risque, l'équité passe avant la performance brute, puis on privilégie la sobriété.

1. On écarte les modèles **sans correction**, et ceux qui ne prédisent presque jamais >50K (taux prédit < 5 %) ;
2. parmi les autres, on garde ceux dont le F1 macro est à moins de **0,03** du meilleur ;
3. on choisit celui qui a le plus petit **écart de parité sexe × origine** ; en cas d'égalité, le moins de paramètres entraînés.

In [ ]:
candidates = results[~results.index.str.contains("original")]
candidates = candidates[candidates["positive_rate_pred"] >= 0.05]
if candidates.empty:
    raise RuntimeError("Aucun modèle corrigé exploitable : augmenter EPOCHS ou le learning rate.")
best_f1 = candidates["f1_macro"].max()
shortlist = candidates[candidates["f1_macro"] >= best_f1 - 0.03]
chosen_name = shortlist.sort_values(
    ["sex_race_demographic_parity_gap", "trainable_parameters"]
).index[0]

print("Short-list :", list(shortlist.index))
print("Modèle retenu :", chosen_name)
display(results.loc[[chosen_name], cols].round(4))

chosen_model = {
    "Full FT (fair)": model_full_fair,
    "LoRA (fair)": model_lora_fair,
    "LoRA (fair, sans attributs)": model_lora_blind,
    "Distillation (fair)": student_model,
}[chosen_name]
chosen_is_blind = chosen_name == "LoRA (fair, sans attributs)"

## 16. Fonction de prédiction pour l'XAI

In [ ]:
def predict_proba(texts, model=None):
    model = chosen_model if model is None else model
    model.eval()
    batch = tokenizer(
        list(texts), padding=True, truncation=True,
        max_length=MAX_LENGTH, return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        logits = model(**batch).logits
    return torch.softmax(logits.float(), dim=-1).cpu().numpy()

text_col = "text_blind" if chosen_is_blind else "text"
predict_proba(test_df[text_col].head(3).tolist())

## 17. Démo : le modèle comme solution (red teaming contrefactuel)

Un conseiller saisit le dossier d'un demandeur. On garde **exactement le même profil** et on ne change que le sexe et l'origine. Si le modèle est équitable, la probabilité ne devrait (presque) pas bouger. On compare le modèle **avant** correction et le modèle **retenu**.

In [ ]:
base_profile = dict(age=40, workclass="Private", education="Bachelors",
                    occupation="Exec-managerial", **{"hours-per-week": 45})
combos = [("Male", "White"), ("Female", "White"), ("Male", "Black"), ("Female", "Black")]
profiles = [{**base_profile, "sex": s, "race": r} for s, r in combos]

rows = []
for p in profiles:
    full_text = row_to_text(p)
    rows.append({
        "profil": f"{p['sex']} | {p['race']}",
        "P(>50K) avant (LoRA original)": predict_proba([full_text], model_lora_raw)[0, 1],
        f"P(>50K) après ({chosen_name})": predict_proba(
            [row_to_text(p, blind=chosen_is_blind)], chosen_model)[0, 1],
    })
demo = pd.DataFrame(rows).set_index("profil")
display(demo.style.format("{:.1%}"))
print("Écart max avant :", f"{demo.iloc[:, 0].max() - demo.iloc[:, 0].min():.1%}",
      "| après :", f"{demo.iloc[:, 1].max() - demo.iloc[:, 1].min():.1%}")

demo.plot(kind="barh", figsize=(9, 4), title="Même dossier, seuls le sexe et l'origine changent")
plt.xlabel("Probabilité prédite >50K")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/demo_contrefactuelle.png", dpi=160)
plt.show()

demo.to_csv(f"{RESULTS_DIR}/demo_contrefactuelle.csv")

## 18. XAI n°1 : LIME

LIME perturbe localement le texte (en retirant des mots) et ajuste un modèle linéaire simple pour estimer quels mots ont le plus influencé **une** prédiction. On l'applique au profil « Female | Black » de la démo, avec le modèle **avant** correction puis le modèle **retenu**.

In [ ]:
from lime.lime_text import LimeTextExplainer

lime_explainer = LimeTextExplainer(class_names=["<=50K", ">50K"])
demo_profile = profiles[3]

lime_tables = {}
for label, model, blind in [("avant", model_lora_raw, False), ("apres", chosen_model, chosen_is_blind)]:
    text = row_to_text(demo_profile, blind=blind)
    exp = lime_explainer.explain_instance(
        text, lambda t, m=model: predict_proba(t, m), num_features=10, num_samples=1000,
    )
    lime_tables[label] = pd.DataFrame(exp.as_list(), columns=["mot", "contribution"])
    exp.save_to_file(f"{RESULTS_DIR}/lime_{label}.html")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (label, tab) in zip(axes, lime_tables.items()):
    tab = tab.sort_values("contribution")
    ax.barh(tab["mot"], tab["contribution"],
            color=["#C44E52" if c < 0 else "#55A868" for c in tab["contribution"]])
    ax.set_title(f"LIME : {label} correction")
    ax.set_xlabel("Contribution vers >50K")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/lime_avant_apres.png", dpi=160)
plt.show()

## 19. XAI n°2 : SHAP

SHAP attribue à chaque fragment de texte une contribution (valeur de Shapley) par rapport à une valeur de référence. Le modèle renvoie les deux probabilités ; on affiche la contribution vers la classe **>50K**.

In [ ]:
import shap

masker = shap.maskers.Text(tokenizer)

def shap_for(model, texts):
    explainer = shap.Explainer(
        lambda t: predict_proba(list(t), model),
        masker,
        output_names=["<=50K", ">50K"],
    )
    return explainer(texts)

shap_before = shap_for(model_lora_raw, [row_to_text(demo_profile)])
shap_after = shap_for(chosen_model, [row_to_text(demo_profile, blind=chosen_is_blind)])

print("Avant correction")
shap.plots.text(shap_before[0, :, ">50K"])
print(f"Après correction ({chosen_name})")
shap.plots.text(shap_after[0, :, ">50K"])

shap.plots.bar(shap_before[0, :, ">50K"], max_display=12, show=False)
plt.title("SHAP : avant correction")
plt.savefig(f"{RESULTS_DIR}/shap_avant.png", dpi=160, bbox_inches="tight")
plt.show()
shap.plots.bar(shap_after[0, :, ">50K"], max_display=12, show=False)
plt.title("SHAP : après correction")
plt.savefig(f"{RESULTS_DIR}/shap_apres.png", dpi=160, bbox_inches="tight")
plt.show()

## 20. Lecture éthique des résultats

Points à commenter après exécution :

- **Performance** : quel modèle offre le meilleur F1 macro ? Le taux de prédiction >50K est-il réaliste ?
- **Équité** : la repondération réduit-elle les écarts par sexe, par origine **et** sur leur croisement ? Supprimer les attributs (variante « sans attributs ») suffit-il, ou le biais passe-t-il par des variables corrélées ?
- **Compromis** : la baisse éventuelle de performance est-elle proportionnée au gain d'équité ?
- **Sobriété** (cours 3) : LoRA et la distillation réduisent-ils les paramètres entraînés, le temps et la taille ?
- **Explicabilité** : LIME et SHAP font-ils ressortir « Female », « Black » ou des variables corrélées ?

**Les 4 types de biais du cours 2 dans ce projet**

| Type | Dans ce projet |
|---|---|
| Données | Écarts historiques de revenu selon le sexe et l'origine (recensement de 1994) |
| Algorithmique | Choix de la cible (le revenu comme indicateur de solvabilité), attributs sensibles dans l'entrée |
| Évaluation | Accuracy trompeuse sur données déséquilibrées : d'où F1, taux prédit et écarts par groupe |
| Interaction / usage | Boucle de rétroaction si le modèle est déployé : refus de crédit → revenus plus faibles → nouveaux refus |

**Amont / pendant / aval (cours 2)**

- *Amont* : audit des données, y compris intersectionnel ;
- *Pendant* : repondération, métriques par sous-groupe ;
- *Aval* (à prévoir en cas de déploiement) : explication fournie au demandeur, procédure de recours humain, suivi des écarts dans le temps.

**Limites**

- Une seule graine aléatoire et 750 exemples de test : les écarts entre modèles peuvent venir du hasard ;
- certains groupes (ex. `Female | Amer-Indian-Eskimo`) sont trop petits pour être évalués ;
- Adult Income est ancien, américain, et reflète des inégalités historiques ;
- une métrique d'équité plus faible ne rend pas le système juste : le choix de la cible, le contexte d'usage et les dommages possibles restent déterminants.

## 21. Trame de présentation (10 minutes)

1. Problème et cas d'usage (crédit, AI Act haut risque) : 1 min
2. Dataset et variables sensibles : 45 s
3. Audit des biais, dont sexe × origine : 1 min 15
4. Méthode de repondération : 45 s
5. Trois méthodes d'adaptation : 1 min
6. Benchmark et modèle retenu : 1 min 30
7. Comparaison avant / après : 1 min
8. Démo contrefactuelle + LIME + SHAP : 1 min 30
9. Types de biais, limites et conclusion : 1 min 15

Remplacer dans les diapositives tous les résultats provisoires par les valeurs réellement produites dans Colab.